In [2]:
"""
Quantitative Comparison between XDeepSpecT and SHAP
Outputs only the metrics needed for the paper table
"""

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# ============================================
# STEP 1: Define Features from Your Results
# ============================================

# All features that appear in your SHAP analysis
all_features = [
    'rolling_mean', 'lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag7', 'lag8', 'lag9',
    'lag10', 'lag11', 'lag12', 'lag13', 'lag14', 'lag15', 'lag16', 'lag17', 'lag18', 'lag19',
    'lag20', 'lag21', 'lag22', 'lag23', 'lag24', 'temperature'
]

# Features in XDeepSpecT Best Co-cluster (from your heatmap)
xdeepspec_features = ['temperature', 'rolling_mean', 'lag19', 'lag20', 'lag21', 'lag22', 'lag23', 'lag24']

# SHAP ranking on FULL dataset (from your first image)
# Ordered by importance (1 = most important)
shap_full_ranking = {
    'rolling_mean': 1,
    'lag17': 2,
    'lag21': 3,
    'lag18': 4,
    'lag3': 5,
    'lag14': 6,
    'lag16': 7,
    'lag20': 8,
    'lag11': 9,
    'lag10': 10,
    'lag9': 11,
    'lag2': 12,
    'lag7': 13,
    'lag19': 14,
    'lag8': 15,
    'lag5': 16,
    'lag23': 17,
    'lag22': 18,
    'lag4': 19,
    'lag1': 20
}

# SHAP ranking on BEST CO-CLUSTER (from your third image)
# Ordered by importance (1 = most important)
shap_cocluster_ranking = {
    'rolling_mean': 1,
    'lag19': 2,
    'temperature': 3,
    'lag21': 4,
    'lag20': 5,
    'lag23': 6,
    'lag22': 7,
    'lag24': 8
}

# XDeepSpecT ranking (features in co-cluster get top ranks)
xdeepspec_ranking = {}
rank = 1
for feat in xdeepspec_features:
    xdeepspec_ranking[feat] = rank
    rank += 1
# Features not in co-cluster get lower ranks
for feat in all_features:
    if feat not in xdeepspec_features:
        xdeepspec_ranking[feat] = rank
        rank += 1

# ============================================
# STEP 2: Compute Spearman's Rank Correlation
# ============================================

# Get ranks for each feature in the same order
features_list = all_features

# Create rank lists
xdeepspec_ranks = [xdeepspec_ranking.get(f, 999) for f in features_list]
shap_full_ranks = [shap_full_ranking.get(f, 999) for f in features_list]
shap_cocluster_ranks = [shap_cocluster_ranking.get(f, 999) for f in features_list]

# Compute Spearman's Rho (only for XDeepSpecT comparisons)
spearman_xd_full, _ = spearmanr(xdeepspec_ranks, shap_full_ranks)
spearman_xd_cocluster, _ = spearmanr(xdeepspec_ranks, shap_cocluster_ranks)

# ============================================
# STEP 3: Compute Top-k Agreement
# ============================================

def top_k_agreement(ranks1, ranks2, k=5):
    """Calculate percentage of top-k features that overlap"""
    top1 = [f for f, r in ranks1.items() if r <= k]
    top2 = [f for f, r in ranks2.items() if r <= k]
    overlap = len(set(top1) & set(top2))
    return (overlap / k) * 100

top5_xd_full = top_k_agreement(xdeepspec_ranking, shap_full_ranking, 5)
top5_xd_cocluster = top_k_agreement(xdeepspec_ranking, shap_cocluster_ranking, 5)

top10_xd_full = top_k_agreement(xdeepspec_ranking, shap_full_ranking, 10)
top10_xd_cocluster = top_k_agreement(xdeepspec_ranking, shap_cocluster_ranking, 10)

# ============================================
# STEP 4: Lag Feature Pattern Analysis
# ============================================

# XDeepSpecT lag features (contiguous block)
xdeepspec_lags = [f for f in xdeepspec_features if 'lag' in f]
xdeepspec_lag_numbers = sorted([int(f.split('lag')[1]) for f in xdeepspec_lags])

# SHAP full dataset lag features (top 15)
shap_full_lags = [f for f, r in shap_full_ranking.items() if 'lag' in f and r <= 15]
shap_full_lag_numbers = sorted([int(f.split('lag')[1]) for f in shap_full_lags])

# Create pattern descriptions
xdeepspec_pattern = f"Contiguous (lag{min(xdeepspec_lag_numbers)}--lag{max(xdeepspec_lag_numbers)})"

# Create SHAP scattered pattern description (first few and last few)
if len(shap_full_lag_numbers) > 6:
    first_few = shap_full_lag_numbers[:3]
    last_few = shap_full_lag_numbers[-3:]
    shap_pattern = f"Scattered (lag{first_few[0]}, lag{first_few[1]}, lag{first_few[2]}, ..., lag{last_few[-3]}, lag{last_few[-2]}, lag{last_few[-1]})"
else:
    shap_pattern = f"Scattered ({', '.join([f'lag{l}' for l in shap_full_lag_numbers])})"

# ============================================
# STEP 5: Print Results in Clean Format
# ============================================

print("=" * 70)
print("QUANTITATIVE COMPARISON: XDeepSpecT vs SHAP")
print("=" * 70)
print()
print("Spearman's ρ (XDeepSpecT vs SHAP full):       ", f"{spearman_xd_full:.3f}")
print("Spearman's ρ (XDeepSpecT vs SHAP co-cluster): ", f"{spearman_xd_cocluster:.3f}")
print()
print("Top-5 Agreement (XDeepSpecT vs SHAP full):     ", f"{top5_xd_full:.1f}%")
print("Top-5 Agreement (XDeepSpecT vs SHAP co-cluster):", f"{top5_xd_cocluster:.1f}%")
print()
print("Top-10 Agreement (XDeepSpecT vs SHAP full):    ", f"{top10_xd_full:.1f}%")
print("Top-10 Agreement (XDeepSpecT vs SHAP co-cluster):", f"{top10_xd_cocluster:.1f}%")
print()
print("XDeepSpecT Lag Pattern:                        ", xdeepspec_pattern)
print("SHAP (full dataset) Lag Pattern:               ", shap_pattern)

# ============================================
# STEP 6: Generate LaTeX Table
# ============================================

print("\n" + "=" * 70)
print("LATEX TABLE FOR YOUR PAPER")
print("=" * 70)

latex_table = f"""
\\begin{{table}}[h]
\\centering
\\caption{{Quantitative comparison of feature importance between XDeepSpecT and SHAP}}
\\label{{tab:quantitative_shap}}
\\begin{{tabular}}{{lcc}}
\\hline
Metric & XDeepSpecT vs SHAP (full) & XDeepSpecT vs SHAP (co-cluster) \\\\
\\hline
Spearman's $\\rho$ & {spearman_xd_full:.3f} & {spearman_xd_cocluster:.3f} \\\\
Top-5 Agreement (\\%) & {top5_xd_full:.1f}\\% & {top5_xd_cocluster:.1f}\\% \\\\
Top-10 Agreement (\\%) & {top10_xd_full:.1f}\\% & {top10_xd_cocluster:.1f}\\% \\\\
\\hline
\\multicolumn{{3}}{{l}}{{\\textit{{Lag Feature Pattern}}}} \\\\
XDeepSpecT & \\multicolumn{{2}}{{c}}{{{xdeepspec_pattern}}} \\\\
SHAP (full dataset) & \\multicolumn{{2}}{{c}}{{{shap_pattern}}} \\\\
\\hline
\\end{{tabular}}
\\footnotesize{{Note: Spearman's $\\rho$ measures rank correlation between feature importance rankings. Top-k agreement indicates the percentage of top-k features that overlap between methods.}}
\\end{{table}}
"""

print(latex_table)

# # ============================================
# # STEP 7: Generate Interpretation Text
# # ============================================

# print("\n" + "=" * 70)
# print("INTERPRETATION TEXT FOR YOUR PAPER")
# print("=" * 70)

# interpretation = f"""
# \\textbf{{Key Findings:}}

# \\begin{{enumerate}}
#     \\item \\textbf{{Stronger agreement with co-cluster SHAP:}} The Spearman correlation between XDeepSpecT and SHAP applied to the best co-cluster ($\\rho = {spearman_xd_cocluster:.3f}$) is substantially higher than with SHAP applied to the full dataset ($\\rho = {spearman_xd_full:.3f}$). This confirms that XDeepSpecT effectively isolates the most relevant feature subset, while global SHAP is influenced by noise and locally relevant features.
    
#     \\item \\textbf{{Perfect top-5 agreement with co-cluster SHAP:}} There is {top5_xd_cocluster:.0f}\\% agreement on the top-5 most important features between XDeepSpecT and SHAP applied to the co-cluster. Both methods consistently identify \\texttt{{rolling\\_mean}}, \\texttt{{lag19}}, \\texttt{{lag20}}, \\texttt{{lag21}}, and \\texttt{{lag22}} as the most critical predictors for ozone forecasting.
    
#     \\item \\textbf{{Contiguous versus scattered lag patterns:}} XDeepSpecT identifies a contiguous block of lag features ({xdeepspec_pattern}), representing a full daily cycle that aligns with the known diurnal cycle of ozone formation. In contrast, SHAP applied to the full dataset identifies scattered lag features {shap_pattern} that lack temporal coherence and do not correspond to meaningful physical patterns.
    
#     \\item \\textbf{{Weak correlation with global SHAP:}} The negative correlation ($\\rho = {spearman_xd_full:.3f}$) and low top-5 agreement ({top5_xd_full:.0f}\\%) indicate that XDeepSpecT and global SHAP identify fundamentally different feature sets. This demonstrates that XDeepSpecT provides complementary interpretability by focusing on temporally-coherent feature groups rather than isolated features.
# \\end{{enumerate}}
# """

# print(interpretation)

# ============================================
# STEP 8: Save Results to CSV (Optional)
# ============================================

# Create summary for CSV export
summary_data = {
    'Metric': [
        'Spearman_rho_XD_vs_SHAP_full',
        'Spearman_rho_XD_vs_SHAP_cocluster',
        'Top5_Agreement_XD_vs_SHAP_full',
        'Top5_Agreement_XD_vs_SHAP_cocluster',
        'Top10_Agreement_XD_vs_SHAP_full',
        'Top10_Agreement_XD_vs_SHAP_cocluster',
        'XD_Lag_Pattern',
        'SHAP_Full_Lag_Pattern'
    ],
    'Value': [
        f"{spearman_xd_full:.3f}",
        f"{spearman_xd_cocluster:.3f}",
        f"{top5_xd_full:.1f}%",
        f"{top5_xd_cocluster:.1f}%",
        f"{top10_xd_full:.1f}%",
        f"{top10_xd_cocluster:.1f}%",
        xdeepspec_pattern,
        shap_pattern
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('xdeepspec_shap_comparison.csv', index=False)
print("\nResults saved to: xdeepspec_shap_comparison.csv")

QUANTITATIVE COMPARISON: XDeepSpecT vs SHAP

Spearman's ρ (XDeepSpecT vs SHAP full):        -0.109
Spearman's ρ (XDeepSpecT vs SHAP co-cluster):  0.814

Top-5 Agreement (XDeepSpecT vs SHAP full):      40.0%
Top-5 Agreement (XDeepSpecT vs SHAP co-cluster): 100.0%

Top-10 Agreement (XDeepSpecT vs SHAP full):     30.0%
Top-10 Agreement (XDeepSpecT vs SHAP co-cluster): 80.0%

XDeepSpecT Lag Pattern:                         Contiguous (lag19--lag24)
SHAP (full dataset) Lag Pattern:                Scattered (lag2, lag3, lag7, ..., lag19, lag20, lag21)

LATEX TABLE FOR YOUR PAPER

\begin{table}[h]
\centering
\caption{Quantitative comparison of feature importance between XDeepSpecT and SHAP}
\label{tab:quantitative_shap}
\begin{tabular}{lcc}
\hline
Metric & XDeepSpecT vs SHAP (full) & XDeepSpecT vs SHAP (co-cluster) \\
\hline
Spearman's $\rho$ & -0.109 & 0.814 \\
Top-5 Agreement (\%) & 40.0\% & 100.0\% \\
Top-10 Agreement (\%) & 30.0\% & 80.0\% \\
\hline
\multicolumn{3}{l}{\textit{Lag Feature 